**dayofyear():**
- is used to **extract** the **day of the year** (an **integer** between **1 and 365/ 366**) from a given **date or timestamp column**.
- `Returns day number in year (1–365/366)`.
  - `Jan 1 → 1`
  - `Dec 31 → 365 (or 366 in leap year)`
- This function has been available **since Spark 1.5**. 

**Leap Years:**
- `considers leap year automatically`.
- The function correctly accounts for **leap years**, returning **366 for December 31st in years like 2024**.

##### Syntax

     dayofyear(expr)

|  Argument  |               Description                                    |
|------------|--------------------------------------------------------------|
| **expr**   | **date / timestamp** column                                  |
| **Data Types** | The `input column` should be of **DateType or TimestampType**. If it is a **string**, you should **first convert** it using **to_date() or to_timestamp()**. |
| **Return** | **day of the year** for given **date/timestamp as integer**. |

In [0]:
from pyspark.sql.functions import dayofyear, expr, to_date

##### 1) Getting the `dayofyear` of `date string column`

In [0]:
df_str_yr = spark.createDataFrame([["Sruthi", "2025-01-01"],
                                   ["Barath", "2025-01-31"],
                                   ["Ravi", "2025-03-10"],
                                   ["Sharma", "2025-05-20"],
                                   ["Kiran", "2025-07-15"],
                                   ["Swaroop", "2025-11-20"],
                                   ["Vikram", "2024-12-31"],
                                   ["Senthil", "2025-12-31"]],
                                  ["name", "DOJ"])
display(df_str_yr)

name,DOJ
Sruthi,2025-01-01
Barath,2025-01-31
Ravi,2025-03-10
Sharma,2025-05-20
Kiran,2025-07-15
Swaroop,2025-11-20
Vikram,2024-12-31
Senthil,2025-12-31


In [0]:
# dayofyear works on both "string date" and "date" columns
df_str_yr \
    .withColumn("DOJ", to_date("DOJ")) \
        .select("DOJ", dayofyear("DOJ").alias("day")).display()

DOJ,day
2025-01-01,1
2025-01-31,31
2025-03-10,69
2025-05-20,140
2025-07-15,196
2025-11-20,324
2024-12-31,366
2025-12-31,365


##### 2) Getting the `dayofyear` using `current_date`

In [0]:
from pyspark.sql.functions import current_date, dayofyear

df_update = spark.range(1).select(current_date().alias("today"), dayofyear(current_date()).alias("dayofyear"))
display(df_update)

today,dayofyear
2026-03-19,78


##### 3) Using `dayofyear` with a `date Column`

In [0]:
import datetime
df_dt = spark.createDataFrame([("Kiran", datetime.date(2015, 4, 8),),
                               ("Amar", datetime.date(2024, 10, 31),),
                               ("Santhosh", datetime.date(2015, 1, 1),),
                               ("Nitin", datetime.date(2015, 12, 31),),
                               ("Rupesh", datetime.date(2013, 12, 31),)],
                               ["name", "birthday"])

display(df_dt)

name,birthday
Kiran,2015-04-08
Amar,2024-10-31
Santhosh,2015-01-01
Nitin,2015-12-31
Rupesh,2013-12-31


In [0]:
df_dt_final = df_dt.select("*", expr("typeof(birthday)").alias("type"), dayofyear('birthday').alias("year"))
display(df_dt_final)

name,birthday,type,year
Kiran,2015-04-08,date,98
Amar,2024-10-31,date,305
Santhosh,2015-01-01,date,1
Nitin,2015-12-31,date,365
Rupesh,2013-12-31,date,365


##### 4) Using `dayofmonth` with a `timestamp Column`

**a) dayofmonth with a timestamp Column using datetime**

In [0]:
import datetime
df_ts = spark.createDataFrame([(datetime.datetime(2015, 4, 8, 13, 8, 15),),
                               (datetime.datetime(2024, 10, 31, 10, 9, 16),),
                               (datetime.datetime(2015, 1, 1, 11, 10, 17),),
                               (datetime.datetime(2015, 12, 31, 12, 11, 18),),
                               (datetime.datetime(2014, 12, 31, 13, 12, 19),),], ['ts'])
                            
display(df_ts)

ts
2015-04-08T13:08:15.000Z
2024-10-31T10:09:16.000Z
2015-01-01T11:10:17.000Z
2015-12-31T12:11:18.000Z
2014-12-31T13:12:19.000Z


In [0]:
df_ts_final = df_ts.select("*", expr("typeof(ts)").alias("type"), dayofyear('ts').alias("year"))
display(df_ts_final)

ts,type,year
2015-04-08T13:08:15.000Z,timestamp,98
2024-10-31T10:09:16.000Z,timestamp,305
2015-01-01T11:10:17.000Z,timestamp,1
2015-12-31T12:11:18.000Z,timestamp,365
2014-12-31T13:12:19.000Z,timestamp,365


**b) dayofmonth with a timestamp Column using to_timestamp**

In [0]:
from pyspark.sql.functions import dayofyear, to_timestamp

df_ts_01 = spark.createDataFrame([("2024-03-17 14:30:00",),
                                  ("2025-06-27 16:45:55",),
                                  ("2026-09-29 18:50:50",),
                                  ("2027-12-31 20:55:55",),
                                  ("2028-01-01 22:00:00",)], ["timestamp"]) \
  .withColumn("ts", to_timestamp("timestamp"))

display(df_ts_01)

timestamp,ts
2024-03-17 14:30:00,2024-03-17T14:30:00.000Z
2025-06-27 16:45:55,2025-06-27T16:45:55.000Z
2026-09-29 18:50:50,2026-09-29T18:50:50.000Z
2027-12-31 20:55:55,2027-12-31T20:55:55.000Z
2028-01-01 22:00:00,2028-01-01T22:00:00.000Z


In [0]:
df_ts_final = df_ts_01.select("ts", dayofyear("ts").alias("day_of_year"))
display(df_ts_final)

ts,day_of_year
2024-03-17T14:30:00.000Z,77
2025-06-27T16:45:55.000Z,178
2026-09-29T18:50:50.000Z,272
2027-12-31T20:55:55.000Z,365
2028-01-01T22:00:00.000Z,1


##### 5) Filtering Data
- `Gets all records in first 100 days of year`

In [0]:
%sql
CREATE OR REPLACE TABLE events (
    id INT,
    date DATE
);

INSERT INTO events VALUES
(1, DATE '2024-01-01'),   -- Day 1
(2, DATE '2024-02-15'),   -- Day 46
(3, DATE '2024-03-31'),   -- Day 91
(4, DATE '2024-04-10'),   -- Day 101 ❌ (outside)
(5, DATE '2024-05-01'),   -- Day 122 ❌
(6, DATE '2024-12-31');   -- Day 366 ❌ (leap year)

SELECT * FROM events;

id,date
1,2024-01-01
2,2024-02-15
3,2024-03-31
4,2024-04-10
5,2024-05-01
6,2024-12-31


In [0]:
df_fltr = spark.table("events")

df_fltr_col = df_fltr.withColumn("day", dayofyear("date"))
display(df_fltr_col)

id,date,day
1,2024-01-01,1
2,2024-02-15,46
3,2024-03-31,91
4,2024-04-10,101
5,2024-05-01,122
6,2024-12-31,366


In [0]:
df_fltr_final = df_fltr_col.filter(dayofyear("date") <= 100)
display(df_fltr_final)

id,date,day
1,2024-01-01,1
2,2024-02-15,46
3,2024-03-31,91


##### 6) With Condition

In [0]:
%sql
CREATE OR REPLACE TABLE sales_dates (
    id INT,
    date DATE
);

INSERT INTO sales_dates VALUES
(1, DATE '2024-01-10'),
(2, DATE '2024-03-31'),
(3, DATE '2024-04-15'),
(4, DATE '2024-06-29'),
(5, DATE '2024-07-01'),
(6, DATE '2024-12-31');

SELECT * FROM sales_dates;

id,date
1,2024-01-10
2,2024-03-31
3,2024-04-15
4,2024-06-29
5,2024-07-01
6,2024-12-31


In [0]:
from pyspark.sql.functions import when

df_when = spark.table("sales_dates")

df_when_final = df_when \
    .withColumn("day", dayofyear("date")) \
    .withColumn("quarter_flag", when(dayofyear("date") <= 90, "Q1")
                                .when(dayofyear("date") <= 180, "Q2")
                                .otherwise("Later"))

display(df_when_final)

id,date,day,quarter_flag
1,2024-01-10,10,Q1
2,2024-03-31,91,Q2
3,2024-04-15,106,Q2
4,2024-06-29,181,Later
5,2024-07-01,183,Later
6,2024-12-31,366,Later
